In [4]:
import eval_beir
from redis_bm25 import gather_bm25_results
from redis_vector import gather_res_vector
from beir.retrieval.evaluation import EvaluateRetrieval

/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Eval beir datasets against Redis with different search techniques

### Load dataset

In [5]:
from redisvl.utils.vectorize import HFTextVectorizer

# set vectorizer
emb_model = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# set dataset of interest
dataset = "fiqa"

corpus, queries, qrels = eval_beir.get_beir_dataset(dataset)
corpus_data = eval_beir.process_corpus(corpus, emb_model=emb_model)

NameError: name 'eval_beir_helpers' is not defined

In [4]:
index = eval_beir_helpers.create_redis_index(dataset)

Index already exists with: 57638 documents


In [ ]:
# load if need be
index.load(corpus_data)

### Check index info

See if all docs indexed, if there were any indexing errors, and view the total indexing time.

In [5]:
num_docs = index.info()["num_docs"]
total_indexing_time = round(float(index.info()["total_indexing_time"]) / 1000, 3) # indexing time in ms
indexing_failures = index.info()['Index Errors']

print(f"{num_docs=}, {total_indexing_time=} s, {indexing_failures=}")

num_docs=57638, total_indexing_time=81.129 s, indexing_failures=['indexing failures', 0, 'last indexing error', 'N/A', 'last indexing error key', 'N/A']


## Eval for different search methods

In [12]:
# simple bm25

redis_res_bm25 = gather_bm25_results(queries, index)
redis_bm25_ndcg, _map, redis_bm25_recall, redis_bm25_precision = (
    EvaluateRetrieval.evaluate(qrels, redis_res_bm25, [1, 10])
)

print(f"{redis_bm25_ndcg=} \n {redis_bm25_recall=} \n {redis_bm25_precision=}")

redis_bm25_ndcg={'NDCG@1': 0.2037, 'NDCG@10': 0.23083} 
 redis_bm25_recall={'Recall@1': 0.1057, 'Recall@10': 0.29653} 
 redis_bm25_precision={'P@1': 0.2037, 'P@10': 0.0642}


In [13]:
# basic vector

redis_res_vector = gather_res_vector(queries, index, emb_model)
vec_ndcg, _map, vec_recall, vec_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_vector, [1, 10]
)

print(f"{vec_ndcg=} \n {vec_recall=} \n {vec_precision=}")

vec_ndcg={'NDCG@1': 0.32716, 'NDCG@10': 0.34469} 
 vec_recall={'Recall@1': 0.15786, 'Recall@10': 0.41133} 
 vec_precision={'P@1': 0.32716, 'P@10': 0.09985}


In [ ]:
# lin combo
from redis_lin_combo import gather_lin_combo_results

alpha = 0.7
redis_res_lin_combo = gather_lin_combo_results(queries, index, alpha, emb_model)
lin_combo_ndcg, _map, lin_combo_recall, lin_combo_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_lin_combo, [1, 10]
)

print(f"{lin_combo_ndcg=} \n {lin_combo_recall=} \n {lin_combo_precision=}")

BM25: lin_combo_ndcg={'NDCG@1': 0.29784, 'NDCG@10': 0.32969} 
 lin_combo_recall={'Recall@1': 0.14983, 'Recall@10': 0.4149} 
 lin_combo_precision={'P@1': 0.29784, 'P@10': 0.10031}


In [ ]:
# weighted rrf
from redis_weighted_rrf import gather_weighted_rrf

redis_res_w_rrf = gather_weighted_rrf(queries, index, emb_model)
w_rrf_ndcg, _map, w_rrf_recall, w_rrf_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_w_rrf, [1, 10]
)

print(f"{w_rrf_ndcg=} \n {w_rrf_recall=} \n {w_rrf_precision=}")

BM25: w_rrf_ndcg={'NDCG@1': 0.3287, 'NDCG@10': 0.34911} 
 w_rrf_recall={'Recall@1': 0.16139, 'Recall@10': 0.42786} 
 w_rrf_precision={'P@1': 0.3287, 'P@10': 0.1}


In [10]:
# weighted rrf
from redis_rerank import gather_rerank_results

redis_res_rerank = gather_rerank_results(queries, index, emb_model)
rerank_ndcg, _map, rerank_recall, rerank_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_rerank, [1, 10]
)

print(f"{rerank_ndcg=} \n {rerank_recall=} \n {rerank_precision=}")

rerank_ndcg={'NDCG@1': 0.27469, 'NDCG@10': 0.32813} 
 rerank_recall={'Recall@1': 0.13487, 'Recall@10': 0.43971} 
 rerank_precision={'P@1': 0.27469, 'P@10': 0.10309}


# Save outputs of run

In [14]:
import pandas as pd

res_data = {
    "Model": ["redis_BM25", "redis_vector", "lin_combo", "weighted_rrf", "rerank"],
    "NDCG@1": [redis_bm25_ndcg['NDCG@1'], vec_ndcg['NDCG@1'], lin_combo_ndcg['NDCG@1'], w_rrf_ndcg['NDCG@1'], rerank_ndcg['NDCG@1']],
    "NDCG@10": [redis_bm25_ndcg['NDCG@10'], vec_ndcg['NDCG@10'], lin_combo_ndcg['NDCG@10'], w_rrf_ndcg['NDCG@10'], rerank_ndcg['NDCG@10']],
    "Recall@1": [redis_bm25_recall['Recall@1'], vec_recall['Recall@1'], lin_combo_recall['Recall@1'], w_rrf_recall['Recall@1'], rerank_recall['Recall@1']],
    "Recall@10": [redis_bm25_recall['Recall@10'], vec_recall['Recall@10'], lin_combo_recall['Recall@10'], w_rrf_recall['Recall@10'], rerank_recall['Recall@10']],
    "Precision@1": [redis_bm25_precision['P@1'], vec_precision['P@1'], lin_combo_precision['P@1'], w_rrf_precision['P@1'], rerank_precision['P@1']],
    "Precision@10": [redis_bm25_precision['P@10'], vec_precision['P@10'], lin_combo_precision['P@10'], w_rrf_precision['P@10'], rerank_precision['P@10']]
}

df = pd.DataFrame(res_data)

In [15]:
df.to_csv(f"{dataset}_results.csv", index=False)

## Cleanup

In [16]:
index.delete()